# CFM Auction Volume Challenge

This notebook summarizes the Challenge Data problem **Stock trading: prediction of auction volumes** proposed by **Capital Fund Management (CFM)**.

Official challenge page: https://challengedata.ens.fr/challenges/60

The goal is to understand the financial setting, the prediction target, the feature families, the time-series structure, and the benchmark modeling approach before building local experiments.

## 1. Challenge Identity

- **Provider:** Capital Fund Management (CFM)
- **Domain:** economic sciences / finance
- **Task type:** supervised regression
- **Data type:** tabular time series
- **Level:** basic
- **Start date:** January 4, 2021

Each row describes one stock on one trading day. The model must predict how much value will be exchanged in the closing auction, expressed through a transformed target.

## 2. Market Context: Closing Auctions

Many exchanges end the trading day with a closing auction. Instead of trades occurring continuously, buy and sell interest is aggregated and the stock is exchanged at one auction price.

This matters because the closing auction can be a cheaper and cleaner way to execute large orders than trading continuously during the day. It is also the last opportunity to change a position before the market closes.

Several types of market participants care about the expected auction volume:

- **Day traders and market makers** may want to reduce or close inventory before overnight risk.
- **Asset managers** may need to reach a target position by the end of the day.
- **Execution algorithms** need to know whether enough liquidity is likely to be available in the auction.

The practical question is: **how much trading capacity should we expect in the auction for a given stock on a given day?**

## 3. Prediction Objective

For each stock and day, predict the auction volume. The challenge describes auction volume as the **total value of stock exchanged** in the auction.

Let:

- $p \in \{1, \ldots, 900\}$ denote a stock, represented by `pid`,
- $t$ denote the trading day, represented by `day`,
- $A_{p,t}$ denote the auction volume for stock $p$ on day $t$,
- $V_{p,t}^{\mathrm{day}}$ denote the total traded volume over the 61 observed intraday periods.

The raw economic quantity of interest is the auction participation ratio:

$$
q_{p,t} = \frac{A_{p,t}}{V_{p,t}^{\mathrm{day}}}.
$$

The target is the natural logarithm of this ratio:

$$
y_{p,t} = \log(q_{p,t})
= \log\left(\frac{A_{p,t}}{V_{p,t}^{\mathrm{day}}}\right).
$$

Example: if the auction represents 10% of the observed daily volume, then

$$
y = \log(0.10) \approx -2.3026.
$$

A model produces a prediction $\widehat{y}_{p,t}$, which can be converted back to an auction-volume ratio with:

$$
\widehat{q}_{p,t} = \exp(\widehat{y}_{p,t}).
$$

## 4. Observation Unit and Time Structure

The row-level unit is:

$$
(p, t)
$$

where $p$ is a product ID and $t$ is a chronological day index.

The `day` column is anonymized as an integer, but its ordering is meaningful: day 0 comes before day 1, day 1 before day 2, and so on.

This is a time-series forecasting problem, not a randomly shuffled tabular problem. The public description says the test inputs correspond to days after the training days. That means validation should imitate forecasting the future from the past.

## 5. Data Scale

The public challenge description gives these important scale facts:

- The same **900 stocks** appear in both train and test.
- The training set covers about **800 days**.
- The test set asks for predictions over about **350 later days**.
- Because stock IDs are shared across train and test, stock-specific behavior can be learned.
- Because test days are later than train days, distribution drift over time is a core risk.

The modeling tension is therefore:

- exploit repeated stock identities,
- avoid leaking future information,
- handle non-stationarity in auction behavior.

## 6. Input Features

The challenge states that each input row has **126 input columns**:

$$
2 + 61 + 61 + 2 = 126.
$$

The feature families are:

- `pid`: product ID, representing the stock.
- `day`: chronological day index.
- `abs_ret0`, ..., `abs_ret60`: absolute intraday returns.
- `rel_vol0`, ..., `rel_vol60`: relative intraday volume shares.
- `LS`: undisclosed trade-related quantity.
- `NLV`: undisclosed trade-related quantity.

The exact column spelling may need to be confirmed once the CSV files are downloaded locally, but the public description defines the feature groups above.

## 7. Intraday Return Features

The challenge divides a large part of the trading day into 61 non-overlapping periods of equal duration.

For period $n \in \{0, \ldots, 60\}$, `abs_retn` is the absolute value of the stock return over that period.

If $P_{p,t,n}^{\mathrm{start}}$ is the last known price at the beginning of period $n$ and $P_{p,t,n}^{\mathrm{end}}$ is the price at the end of that period, the signed return would be:

$$
r_{p,t,n} = \frac{P_{p,t,n}^{\mathrm{end}} - P_{p,t,n}^{\mathrm{start}}}{P_{p,t,n}^{\mathrm{start}}}.
$$

The provided feature is the absolute return:

$$
\mathrm{abs\_ret}_{p,t,n} = \left|r_{p,t,n}\right|.
$$

These variables describe intraday volatility and price movement intensity. Since the sign is removed, they measure how much the stock moved, not whether it went up or down.

## 8. Relative Volume Features

The `rel_voln` variables use the same 61 periods as the return variables.

Let $V_{p,t,n}$ be the volume traded during period $n$. The total observed intraday volume over the 61 periods is:

$$
V_{p,t}^{\mathrm{day}} = \sum_{k=0}^{60} V_{p,t,k}.
$$

The relative volume feature for period $n$ is:

$$
\mathrm{rel\_vol}_{p,t,n}
= \frac{V_{p,t,n}}{\sum_{k=0}^{60} V_{p,t,k}}.
$$

Therefore, for each stock-day row:

$$
\sum_{n=0}^{60} \mathrm{rel\_vol}_{p,t,n} = 1
$$

up to missing values or preprocessing details.

These variables describe the intraday volume profile: when during the day trading activity was concentrated.

## 9. `LS` and `NLV`

`LS` and `NLV` are two additional trade-related quantities. Their meanings are intentionally undisclosed in the challenge description.

Modeling implications:

- Treat them as numeric explanatory variables unless the downloaded data suggests otherwise.
- Check missingness, scale, outliers, and correlation with the target.
- Do not assume financial semantics that are not provided.
- Because they are undisclosed, model inspection should focus on empirical behavior rather than interpretation.

## 10. Feature Schema Reference

The following code cell defines the expected feature groups from the public challenge description. It is intentionally lightweight and can be reused later when the data files are available locally.

In [ ]:
N_PERIODS = 61

ID_FEATURES = ["pid", "day"]
ABS_RETURN_FEATURES = [f"abs_ret{n}" for n in range(N_PERIODS)]
REL_VOLUME_FEATURES = [f"rel_vol{n}" for n in range(N_PERIODS)]
UNDISCLOSED_FEATURES = ["LS", "NLV"]

EXPECTED_FEATURES = (
    ID_FEATURES
    + ABS_RETURN_FEATURES
    + REL_VOLUME_FEATURES
    + UNDISCLOSED_FEATURES
)

len(EXPECTED_FEATURES)

Expected result:

$$
|\mathcal{X}| = 126.
$$

If the real CSV files use slightly different names such as `abs_ret0` versus `abs_retn` template notation, the feature grouping logic should be adjusted after inspecting `X_train.columns`.

## 11. Train/Test Setup

The public page states that train and test contain the same 900 stocks, but the test days occur after the training days.

A suitable abstract split is:

$$
\mathcal{D}_{\mathrm{train}}
= \left\{(x_{p,t}, y_{p,t}) : t \leq T_{\mathrm{train}}\right\}
$$

$$
\mathcal{D}_{\mathrm{test}}
= \left\{x_{p,t} : t > T_{\mathrm{train}}\right\}.
$$

This makes random row-wise validation misleading because it can mix future and past days. A better validation strategy is chronological:

- train on earlier days,
- validate on later days,
- repeat through rolling or expanding windows.

## 12. Missing Values

The official benchmark begins by replacing missing values with zero.

A simple preprocessing function is therefore:

$$
\widetilde{x}_{j} =
\begin{cases}
x_j, & \text{if } x_j \text{ is observed}, \\
0, & \text{if } x_j \text{ is missing}.
\end{cases}
$$

This is a benchmark choice, not necessarily the best possible choice. Later experiments can compare:

- zero imputation,
- median imputation by feature,
- stock-wise imputation,
- missingness indicator features.

## 13. Official Benchmark: Linear Model + Residual Boosting

The public benchmark has two stages.

### Stage 1: linear regression without `pid`

After imputing missing values with zero, fit a linear model on all columns except the stock ID:

$$
\widehat{y}^{\mathrm{lin}}_i
= \beta_0 + \sum_{j \in \mathcal{F}\setminus\{\mathrm{pid}\}} \beta_j x_{i,j}.
$$

Here, $i$ indexes a stock-day observation and $\mathcal{F}$ is the feature set.

The residual is:

$$
e_i = y_i - \widehat{y}^{\mathrm{lin}}_i.
$$

### Stage 2: LightGBM model on residuals

Fit a gradient-boosted tree model to predict $e_i$ using all columns, including `pid`:

$$
\widehat{e}_i = g_{\theta}(x_i).
$$

The final prediction is:

$$
\widehat{y}_i
= \widehat{y}^{\mathrm{lin}}_i + \widehat{e}_i.
$$

The benchmark trains the boosted trees with early stopping and uses `sklearn.model_selection.TimeSeriesSplit` so validation reflects future prediction from mostly past data.

## 14. Why the Benchmark Design Makes Sense

The benchmark combines two useful modeling ideas:

1. **Global linear structure:** a linear model captures broad relationships between volatility, intraday volume shape, undisclosed variables, and log auction participation.
2. **Nonlinear stock-specific corrections:** boosted trees can model nonlinear interactions, threshold effects, missingness patterns, and stock-level differences through `pid`.

The residual formulation can be interpreted as:

$$
y_i = f_{\mathrm{global}}(x_i) + f_{\mathrm{residual}}(x_i) + \epsilon_i.
$$

where:

- $f_{\mathrm{global}}$ is the linear baseline,
- $f_{\mathrm{residual}}$ is learned by LightGBM,
- $\epsilon_i$ is unexplained noise.

## 15. Evaluation Notes

The public challenge text identifies the task as regression and describes the benchmark model. The public page content available without login does not explicitly state the leaderboard metric.

For local experimentation, useful validation losses include:

$$
\mathrm{MSE}
= \frac{1}{N}\sum_{i=1}^{N}\left(y_i - \widehat{y}_i\right)^2,
$$

$$
\mathrm{RMSE}
= \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left(y_i - \widehat{y}_i\right)^2},
$$

$$
\mathrm{MAE}
= \frac{1}{N}\sum_{i=1}^{N}\left|y_i - \widehat{y}_i\right|.
$$

Because the target is logarithmic, squared error on $y$ penalizes multiplicative errors in the original auction-volume ratio.

## 16. Main Modeling Challenges

### Non-stationarity

The challenge description explicitly warns that auction volumes can evolve over time. A model trained on old days may underpredict or overpredict later days if auction participation drifts.

A simple drift model could include a time component:

$$
y_{p,t} = \alpha_p + h(t) + u(x_{p,t}) + \epsilon_{p,t},
$$

where $\alpha_p$ is a stock effect and $h(t)$ is a temporal trend.

### Cross-sectional heterogeneity

The same 900 stocks appear in train and test, so stock-specific effects may matter:

$$
y_{p,t} = \alpha_p + \epsilon_{p,t}.
$$

This suggests trying target encoding, stock-level historical averages, or models that can use `pid` directly.

### Intraday shape effects

The 61-period return and volume curves are not just independent variables. Their shapes may matter. For example, heavy late-day volume could contain different information than heavy early-day volume.

Possible engineered summaries:

$$
\mathrm{volatility\_sum}_{p,t} = \sum_{n=0}^{60} \mathrm{abs\_ret}_{p,t,n},
$$

$$
\mathrm{volume\_concentration}_{p,t}
= \sum_{n=0}^{60} \mathrm{rel\_vol}_{p,t,n}^{2},
$$

$$
\mathrm{late\_volume\_share}_{p,t}
= \sum_{n=k}^{60} \mathrm{rel\_vol}_{p,t,n}
$$

for a chosen late-day cutoff $k$.

## 17. Suggested First Experimental Roadmap

A sensible first local workflow is:

1. Load `X_train`, `y_train`, `X_test`, and the sample submission.
2. Verify column names and shapes.
3. Check missing values by feature family.
4. Plot target distribution overall and by `day`.
5. Plot stock-level target means and variances by `pid`.
6. Reproduce the zero-imputation linear baseline.
7. Add chronological validation with `TimeSeriesSplit`.
8. Add LightGBM on either the target directly or on linear-model residuals.
9. Compare global, stock-aware, and time-aware validation results.
10. Generate a first submission only after the local validation setup is trusted.

## 18. Sources

- Official Challenge Data page: https://challengedata.ens.fr/challenges/60
- `TimeSeriesSplit` documentation referenced by the challenge benchmark: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html
- LightGBM project documentation: https://lightgbm.readthedocs.io/